In [1]:
import torchio as tio
import os
import os.path as path

/home/an_inam/miniconda3/envs/MICC/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
fp=f'/storage/an_inam/MR2MR/Data/10_Pat_t1'
modality_1 = '3T_7TReg'
modality_2 = '7T'
mask_dir_name = 'MASK3T_7TReg'

fpp= fp.join('')

In [18]:
fp_placeholder = fp
fp, fp_placeholder

('/storage/an_inam/MR2MR/Data/10_Pat_t1',
 '/storage/an_inam/MR2MR/Data/10_Pat_t1')

In [28]:
fp3=f'{fp_placeholder}/{modality_1}'
fp7=f'{fp_placeholder}/{modality_2}'
fpm=f'{fp_placeholder}/{mask_dir_name}'


with open('/storage/an_inam/MR2MR/Data/10_pat_names.txt', 'r') as fp:
  fns = [line.strip() for line in fp]

postfix='.nii.gz'
fns, fp, "bbb", fp_placeholder, fp3, 'aaa', fpp

(['sub-01_T1w.nii.gz',
  'sub-02_T1w.nii.gz',
  'sub-03_T1w.nii.gz',
  'sub-04_T1w.nii.gz',
  'sub-05_T1w.nii.gz',
  'sub-06_T1w.nii.gz',
  'sub-07_T1w.nii.gz',
  'sub-08_T1w.nii.gz',
  'sub-09_T1w.nii.gz',
  'sub-10_T1w.nii.gz'],
 <_io.TextIOWrapper name='/storage/an_inam/MR2MR/Data/10_pat_names.txt' mode='r' encoding='UTF-8'>,
 'bbb',
 '/storage/an_inam/MR2MR/Data/10_Pat_t1',
 '/storage/an_inam/MR2MR/Data/10_Pat_t1/3T_7TReg',
 'aaa',
 '')

In [22]:
print(f'Number of subjects: {len(fns)}')

Number of subjects: 10


In [30]:
path.join(fp_placeholder,str(fns[0])+postfix), fp3

('/storage/an_inam/MR2MR/Data/10_Pat_t1/sub-01_T1w.nii.gz.nii.gz',
 '/storage/an_inam/MR2MR/Data/10_Pat_t1/3T_7TReg')

In [ ]:
#load all subjects with torchio
subjects=[]
for i in range(len(fns)):
    #normalize value between 0,1
    #outlier removeal done during preprocessing (5%-95% percentile kept)
    rescale1=tio.RescaleIntensity(out_min_max=(0, 1),percentiles=(0,100))
    rescale2=tio.RescaleIntensity(out_min_max=(0, 1),percentiles=(0,100))
    subject=tio.Subject(
        t1_3T = rescale1(tio.ScalarImage(path.join(fp3,str(fns[i])))),
        t1_7T = rescale2(tio.ScalarImage(path.join(fp7,str(fns[i])))),
        mask = tio.LabelMap(path.join(fpm,str(fns[i])))
    )
    subjects.append(subject)
# org_dataset = tio.SubjectsDataset(subjects)
print('Dataset size:', len(subjects), 'subjects')

# data augmentation

In [7]:
transform = tio.Compose([
    # Coordinate changes for 3T & 7T
    tio.RandomFlip(axes=('LR','AP'),
                    p=1,
                    include=('t1_3T','t1_7T', 'mask')),
    tio.RandomAffine(scales=0, #no scaling, data in MNI space
                    degrees=20,
                    isotropic=True,
                    p=1,
                    include=('t1_3T','t1_7T', 'mask')),
    # Elastic deformation may cause some resolution degradation
    tio.RandomElasticDeformation(num_control_points=7,
                    p=1,
                    include=('t1_3T','t1_7T', 'mask')),  
    # image alterations for 3T only
    tio.RandomGamma(log_gamma=(-0.3,0.3),
                    p=1,
                    include=['t1_3T']),
    # allow image downsampling in each dimension (3T only)
    ### Random anisotropy causes shift!
    # do resample and then resample back!
    tio.RandomAnisotropy(axes=(0),
                         downsampling=(1, 5),
                         p=1,
                         include=['t1_3T'],
                         ),
    tio.RandomAnisotropy(axes=(1),
                         downsampling=(1, 5),
                         p=1,
                         include=['t1_3T']),
    tio.RandomAnisotropy(axes=(2),
                         downsampling=(1, 5),
                         p=1,
                         include=['t1_3T']),                                        
])

In [9]:
subject_set = tio.SubjectsDataset(subjects)

In [12]:
fp_placeholder

'/Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin'

In [ ]:
num_transforms_sets = 2  # Define the number of transforms you want to apply
for transform_index in range(num_transforms_sets):
  print(f'{transform_index}/{num_transforms_sets}')
  for i, subject in enumerate(subject_set):
    transformed_subject = transform(subject)
    directory3t = f'{fp_placeholder}/transform_{transform_index}/{modality_1}'
    directory7t = f'{fp_placeholder}/transform_{transform_index}/{modality_2}'
    directorymask = f'{fp_placeholder}/transform_{transform_index}/{mask_dir_name}'

    os.makedirs(directory3t,exist_ok=True)
    os.makedirs(directory7t,exist_ok=True)
    os.makedirs(directorymask,exist_ok=True)
    print(f'{i+1}/{len(subjects)} Saving transformed subjects to {directory3t}, {directory7t}, {directorymask}')
    transformed_subject.t1_3T.save(f'{directory3t}/{fns[i]}_transformed_augmentated_{transform_index}.nii.gz')
    transformed_subject.t1_7T.save(f'{directory7t}/{fns[i]}_transformed_augmentated_{transform_index}.nii.gz')
    transformed_subject.mask.save(f'{directorymask}/{fns[i]}_transformed_augmentated_{transform_index}.nii.gz')

0/2
1/10 Saving transformed subjects to /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/3T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/7T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/mask
2/10 Saving transformed subjects to /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/3T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/7T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/mask
3/10 Saving transformed subjects to /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/3T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/7T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/mask
4/10 Saving transformed subjects to /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/3T_t1, /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_0/7T_t1, /Drive4T/inam/datasets/3T_7T_Data

# downsample

In [16]:
scale_factor = 2
resampled_subjects = []
for subject in subjects:
    spacing = subject.t1_3T.spacing
    new_spacing = [s * scale_factor for s in spacing]
    resample_down = tio.Resample(new_spacing)
    resample_up = tio.Resample(spacing)
    resampled_subject = resample_down(subject)
    resampled_subject = resample_up(resampled_subject)
    resampled_subjects.append(resampled_subject)

In [17]:
print(resampled_subjects)

[Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3), Subject(Keys: ('t1_3T', 't1_7T', 'mask'); images: 3)]


In [19]:
resampled_set = tio.SubjectsDataset(resampled_subjects)
directorydownscale = f'{fp_placeholder}/transform_downscale_{scale_factor}/'
print(f'Downscale directory: {directorydownscale}')
os.makedirs(directorydownscale, exist_ok=True)
os.makedirs(f'{directorydownscale}/{modality_1}/', exist_ok=True)
os.makedirs(f'{directorydownscale}/{modality_2}/', exist_ok=True)
os.makedirs(f'{directorydownscale}/{mask_dir_name}/',exist_ok=True)
  

for i, subject in enumerate(resampled_set):
    subject.t1_3T.save(f'{directorydownscale}/{modality_1}/{fns[i]}_down{scale_factor}.nii.gz')
    subject.t1_7T.save(f'{directorydownscale}/{modality_2}/{fns[i]}_down{scale_factor}.nii.gz')
    subject.mask.save(f'{directorydownscale}/{mask_dir_name}/{fns[i]}_down{scale_factor}.nii.gz')

    

Downscale directory: /Drive4T/inam/datasets/3T_7T_Data/T1/skull_stripped/Fin/transform_downscale_2/
